In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

#**Parameters**

Widgets let us reuse this one notebook for any dimension table (flights, airports, passengers) by passing in different values:
- **keycols**: business key column(s) that identify a record, e.g. `['passenger_id']`
- **cdccols**: change-tracking column used to find new or changed rows, e.g. `modifiedDate`
- **backdated_refresh**: optional date to reload from; leave empty for a normal incremental run
- **surrogate_key_col**: name of the surrogate key column in the dimension table
- **catalog_name / source_schema / source_object**: where the source (silver) data lives
- **target_schema / target_object**: where the dimension (gold) table is written

In [0]:
# Define the parameters

# KEY COLUMNS
dbutils.widgets.text("keycols", "")

# CDC COLUMN
dbutils.widgets.text("cdccols", "")

# BACK DATED REFRESH
dbutils.widgets.text("backdated_refresh", "")

# Surrogate Key Col
dbutils.widgets.text("surrogate_key_col", "")

# Source Object
dbutils.widgets.text("source_object", "")

# Catalog name
dbutils.widgets.text("catalog_name", "databricksete")

# Source Schema
dbutils.widgets.text("source_schema", "")

# Target Object
dbutils.widgets.text("target_object", "")

# Target Schema
dbutils.widgets.text("target_schema", "")

In [0]:
# Fetching Parameters and creating variables

# Key Columns
key_cols_list = eval(dbutils.widgets.get("keycols"))

# CDC Column
cdc_col = dbutils.widgets.get("cdccols").strip()

# Back Dated Refresh
backdated_refresh = dbutils.widgets.get("backdated_refresh").strip()

# Surrogate Key Col
surrogate_key_col = dbutils.widgets.get("surrogate_key_col").strip()

# Catalog name
catalog_name = dbutils.widgets.get("catalog_name").strip()

# Source Object
source_object = dbutils.widgets.get("source_object").strip()

# Source Schema
source_schema = dbutils.widgets.get("source_schema").strip()

# Target Object
target_object = dbutils.widgets.get("target_object").strip()

# Target Schema
target_schema = dbutils.widgets.get("target_schema").strip()

# **Incremental Data Ingestion**

Reads only the rows that are new or changed since the last run, instead of reloading everything.
- Finds `last_load`: the max `cdc_col` already in the target table.
- If the target doesn't exist yet, `last_load` is set to `1900-01-01`, so the first run loads all the data.
- If `backdated_refresh` is given, it overrides `last_load` so we can reload from a set date.
- `df_src` holds the source rows with `cdc_col` greater than `last_load`.

In [0]:
# No backdated refresh
if len(backdated_refresh) == 0:

    # If table exists in the destination, get last load date
    if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):

        last_load = spark.sql(f"""
                              SELECT max({cdc_col})
                              FROM {catalog_name}.{target_schema}.{target_object}
                              """).collect()[0][0]
    
    # If table doesn't exist in the destination, set last load date to 1900-01-01 (COMPLETE LOAD)
    else:
        last_load = "1900-01-01 00:00:00"

# Backdated refresh
else:
    last_load = backdated_refresh

In [0]:
df_src = spark.sql(f"""
                    SELECT * 
                    FROM {catalog_name}.{source_schema}.{source_object}
                    WHERE {cdc_col} > '{last_load}'
                    """)

# **OLD vs NEW Records**

Works out which incoming rows already exist in the dimension table and which are brand new.
- Builds `df_trg` from the target table (keys, surrogate key, `create_date`, `update_date`).
- On the first run the target doesn't exist, so an empty dummy table with the same columns is created (`WHERE 1 = 0`) so the join still works.
- Left joins source to target to retrieve new and old records.
- **Old records**: the surrogate key was found, so the record already exists (an update).
- **New records**: the surrogate key is null, so the record isn't in the target (an insert).

In [0]:
if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):
    
    # Key columns for incremental load
    key_col_string_incremental = ', '.join(key_cols_list)
    
    df_trg = spark.sql(f"""
                        SELECT {key_col_string_incremental}, {surrogate_key_col}, create_date, update_date
                        FROM {catalog_name}.{target_schema}.{target_object}
                        """)
else:

    # Key columns for initial load
    key_col_string_init = [f" '' AS {i}" for i in key_cols_list]
    key_col_string_init = ', '.join(key_col_string_init)

    df_trg = spark.sql(f"""
                        SELECT {key_col_string_init}, CAST('0' AS INT) AS {surrogate_key_col}, CAST('1900-01-01 00:00:00' AS timestamp) AS create_date, CAST('1900-01-01 00:00:00' AS timestamp) AS update_date
                        WHERE 1 = 0
                        """)

 

In [0]:
df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")

# Join condition
join_condition = " AND ".join([f"src.{i} = trg.{i}" for i in key_cols_list])

df_join = spark.sql(f"""
                    SELECT src.*, trg.{surrogate_key_col}, trg.create_date, trg.update_date
                    FROM src
                    LEFT JOIN trg
                    ON {join_condition}
                    """)

In [0]:
# OLD RECORDS
df_old = df_join.filter(col(f'{surrogate_key_col}').isNotNull())

# NEW RECOERDS
df_new = df_join.filter(col(f'{surrogate_key_col}').isNull())

# **Enriching DFs**

Prepares the old and new records for writing by filling in the surrogate key and the audit dates.
- **Old records** already have a surrogate key, so only `update_date` is refreshed to the current timestamp. `create_date` is kept as it was.
- **New records** get a fresh surrogate key: the current max key in the target (or 0 on the first run) + 1 + `monotonically_increasing_id()`. Both `create_date` and `update_date` are set to the current timestamp.

In [0]:
# For old records, set update date
df_old_enr = df_old.withColumn('update_date', current_timestamp())

# For new records, set surrogate key
if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):
    
    maximum_surrogate_key = spark.sql(f"""
                                   SELECT max({surrogate_key_col})
                                   FROM {catalog_name}.{target_schema}.{target_object}
                                   """).collect()[0][0]

    df_new_enr = df_new.withColumn(f'{surrogate_key_col}', lit(maximum_surrogate_key) + lit(1) + monotonically_increasing_id()) \
                   .withColumn('create_date', current_timestamp()) \
                   .withColumn('update_date', current_timestamp())

else:

    maximum_surrogate_key = 0

    df_new_enr = df_new.withColumn(f'{surrogate_key_col}', lit(maximum_surrogate_key) + lit(1) + monotonically_increasing_id()) \
                   .withColumn('create_date', current_timestamp()) \
                   .withColumn('update_date', current_timestamp())

# **Union OLD and NEW Records**

Combines the enriched old (updates) and new (inserts) records into one DataFrame, `df_union`, so a single write handles both.
- `unionByName` matches columns by name rather than position, so column order differences don't cause errors.

In [0]:
df_union = df_old_enr.unionByName(df_new_enr)


# **UPSERT**

Writes `df_union` to the dimension table.
- **Target exists**: runs a Delta `MERGE` on the surrogate key. Matching rows are updated (only if the source `cdc_col` is the same or newer than the target's), and non-matching rows are inserted.
- **Target doesn't exist (first run)**: the table is created by writing `df_union` in append mode with `saveAsTable`.

In [0]:
if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):

    dlt_obj = DeltaTable.forName(spark, f"{catalog_name}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key_col} = src.{surrogate_key_col}")\
                        .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
                        .whenNotMatchedInsertAll()\
                        .execute()

else: 

    df_union.write.format("delta")\
            .mode("append")\
            .saveAsTable(f"{catalog_name}.{target_schema}.{target_object}")
